In [ ]:
import pandas as pd
import pypsa

network_path = "results/networks/base_s_40_elec_.nc"
n = pypsa.Network(network_path)

# 1. Sélectionner la colonne de capacité optimale (ou p_nom si non optimisée)
cap_col = "p_nom_opt" if "p_nom_opt" in n.generators.columns else "p_nom"

# 2. Filtrer spécifiquement sur vos technologies éoliennes
# (adaptez la liste si vous avez d'autres noms comme offwind-ac, etc.)
wind_carriers = ["onwind", "onwind_type2", "offwind-ac", "offwind-dc"]
wind_gens = n.generators[n.generators.carrier.isin(wind_carriers)]

# 3. Calculer la capacité totale installée par technologie (en GW)
wind_capacity = wind_gens.groupby("carrier")[cap_col].sum() / 1e3  # MW -> GW

# 4. Calculer les pourcentages / proportions
wind_share = (wind_capacity / wind_capacity.sum()) * 100

# 5. Calculer la production réelle totale sur la période simulée (en TWh)
if not n.generators_t.p.empty:
    wind_generation = (
        n.generators_t.p[wind_gens.index].sum().T.groupby(wind_gens.carrier).sum()
        / 1e6  # MWh -> TWh
    )
    gen_share = (wind_generation / wind_generation.sum()) * 100
else:
    wind_generation = pd.Series(0, index=wind_capacity.index)
    gen_share = pd.Series(0, index=wind_capacity.index)

# 6. Synthèse sous forme de tableau comparatif
summary_df = pd.DataFrame(
    {
        "Capacité installée (GW)": wind_capacity.round(2),
        "Part de capacité (%)": wind_share.round(1),
        "Production totale (TWh)": wind_generation.round(2),
        "Part de production (%)": gen_share.round(1),
    }
)

print(summary_df)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Camembert des capacités installées
wind_capacity.plot(
    kind="pie",
    ax=ax1,
    autopct="%1.1f%%",
    startangle=90,
    colors=["#38bdf8", "#0284c7", "#0f766e", "#14b8a6"],
)
ax1.set_ylabel("")
ax1.set_title("Répartition de la puissance installée (GW)")

# Bar chart de comparaison
summary_df[["Part de capacité (%)", "Part de production (%)"]].plot(
    kind="bar", ax=ax2, colormap="viridis"
)
ax2.set_ylabel("%")
ax2.set_title("Comparaison : Capacité vs Production réelle")
ax2.grid(axis="y", linestyle="--", alpha=0.6)
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
import pypsa

# Charger votre réseau optimisé
n = pypsa.Network("results/networks/base_s_40_elec_.nc")

# 1. Vérifier la capacité installée optimale (GW) par technologie (notamment onwind_type2)
cap = n.generators.groupby("carrier")["p_nom_opt"].sum() / 1e3
print("--- Capacités optimales (GW) ---")
print(cap.round(2))

# 2. Vérifier l'énergie totale produite (TWh)
p_by_carrier = (
    n.generators_t.p.T.groupby(n.generators.carrier).sum().T / 1e6
)  # TWh
print("\n--- Production totale (TWh) ---")
print(p_by_carrier.sum().round(2))

In [ ]:
#Capacity factor calculations for LW25 and LW13
cf_lw25 = (
    n.generators_t.p_max_pu[
        [g for g in n.generators.index if n.generators.loc[g, "carrier"] == "onwind"]
    ]
    .mean()
    .mean()
)
cf_lw13 = (
    n.generators_t.p_max_pu[
        [
            g
            for g in n.generators.index
            if n.generators.loc[g, "carrier"] == "onwind_type2"
        ]
    ]
    .mean()
    .mean()
)

print(f"Facteur de charge moyen LW25 (onwind) : {cf_lw25:.4f}")
print(f"Facteur de charge moyen LW13 (onwind_type2) : {cf_lw13:.4f}")

In [ ]:
import pandas as pd
import pypsa

network_path = "results/networks/base_s_40_elec_.nc"
n = pypsa.Network(network_path)

mask_t2 = n.generators.carrier == "onwind_type2"
n.generators.loc[mask_t2, "capital_cost"] = (
    n.generators.loc[mask_t2, "capital_cost"] * 0.5
)

# Vérification : la valeur doit maintenant afficher ~59410 €
print(
    "Nouveau capital_cost moyen onwind_type2 :",
    n.generators.loc[mask_t2, "capital_cost"].mean(),
)